In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Retail Sales Dashboard") \
    .getOrCreate()

print(spark.version)

4.0.3


In [3]:
from google.colab import files
uploaded = files.upload()

Saving attendance.csv to attendance.csv
Saving tasks.csv to tasks.csv


In [4]:
attendance_df = spark.read.csv(
    "attendance.csv",
    header=True,
    inferSchema=True
)

tasks_df = spark.read.csv(
    "tasks.csv",
    header=True,
    inferSchema=True
)

In [5]:
attendance_df.show()
tasks_df.show()

+-------------+-----------+---------------+-------------------+-------------------+
|attendance_id|employee_id|attendance_date|           clock_in|          clock_out|
+-------------+-----------+---------------+-------------------+-------------------+
|            1|        101|     2026-06-01|2026-06-01 09:00:00|2026-06-01 18:00:00|
|            2|        102|     2026-06-01|2026-06-01 09:20:00|2026-06-01 17:45:00|
|            3|        103|     2026-06-01|2026-06-01 09:05:00|2026-06-01 18:15:00|
|            4|        104|     2026-06-01|2026-06-01 10:15:00|2026-06-01 17:00:00|
|            5|        105|     2026-06-01|2026-06-01 08:50:00|2026-06-01 17:30:00|
|            6|        106|     2026-06-01|2026-06-01 09:10:00|2026-06-01 18:05:00|
|            7|        107|     2026-06-01|2026-06-01 09:00:00|2026-06-01 18:20:00|
|            8|        108|     2026-06-01|2026-06-01 09:30:00|2026-06-01 17:40:00|
|            9|        109|     2026-06-01|2026-06-01 08:45:00|2026-06-01 18

In [6]:
attendance_df.printSchema()
tasks_df.printSchema()

root
 |-- attendance_id: integer (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- attendance_date: date (nullable = true)
 |-- clock_in: timestamp (nullable = true)
 |-- clock_out: timestamp (nullable = true)

root
 |-- task_id: integer (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- task_name: string (nullable = true)
 |-- tasks_completed: integer (nullable = true)
 |-- task_status: string (nullable = true)



In [7]:
attendance_tasks_df = attendance_df.join(
    tasks_df,
    "employee_id"
)
attendance_tasks_df.show()

+-----------+-------------+---------------+-------------------+-------------------+-------+--------------------+---------------+-----------+
|employee_id|attendance_id|attendance_date|           clock_in|          clock_out|task_id|           task_name|tasks_completed|task_status|
+-----------+-------------+---------------+-------------------+-------------------+-------+--------------------+---------------+-----------+
|        101|            1|     2026-06-01|2026-06-01 09:00:00|2026-06-01 18:00:00|     11|   attendance report|              7|  completed|
|        101|            1|     2026-06-01|2026-06-01 09:00:00|2026-06-01 18:00:00|      1|dashboard develop...|              8|  completed|
|        102|            2|     2026-06-01|2026-06-01 09:20:00|2026-06-01 17:45:00|     12| employee onboarding|              4|  completed|
|        102|            2|     2026-06-01|2026-06-01 09:20:00|2026-06-01 17:45:00|      2|   recruitment drive|              5|  completed|
|        103|

In [8]:
from pyspark.sql.functions import col

attendance_tasks_df = attendance_tasks_df.withColumn(
    "work_hours",
    (
        col("clock_out").cast("timestamp").cast("long") -
        col("clock_in").cast("timestamp").cast("long")
    ) / 3600
)

attendance_tasks_df.show()

+-----------+-------------+---------------+-------------------+-------------------+-------+--------------------+---------------+-----------+-----------------+
|employee_id|attendance_id|attendance_date|           clock_in|          clock_out|task_id|           task_name|tasks_completed|task_status|       work_hours|
+-----------+-------------+---------------+-------------------+-------------------+-------+--------------------+---------------+-----------+-----------------+
|        101|            1|     2026-06-01|2026-06-01 09:00:00|2026-06-01 18:00:00|     11|   attendance report|              7|  completed|              9.0|
|        101|            1|     2026-06-01|2026-06-01 09:00:00|2026-06-01 18:00:00|      1|dashboard develop...|              8|  completed|              9.0|
|        102|            2|     2026-06-01|2026-06-01 09:20:00|2026-06-01 17:45:00|     12| employee onboarding|              4|  completed|8.416666666666666|
|        102|            2|     2026-06-01|202

In [9]:
attendance_tasks_df = attendance_tasks_df.withColumn(
    "effective_work_hours",
    col("work_hours") - 1
)

attendance_tasks_df.show()

+-----------+-------------+---------------+-------------------+-------------------+-------+--------------------+---------------+-----------+-----------------+--------------------+
|employee_id|attendance_id|attendance_date|           clock_in|          clock_out|task_id|           task_name|tasks_completed|task_status|       work_hours|effective_work_hours|
+-----------+-------------+---------------+-------------------+-------------------+-------+--------------------+---------------+-----------+-----------------+--------------------+
|        101|            1|     2026-06-01|2026-06-01 09:00:00|2026-06-01 18:00:00|     11|   attendance report|              7|  completed|              9.0|                 8.0|
|        101|            1|     2026-06-01|2026-06-01 09:00:00|2026-06-01 18:00:00|      1|dashboard develop...|              8|  completed|              9.0|                 8.0|
|        102|            2|     2026-06-01|2026-06-01 09:20:00|2026-06-01 17:45:00|     12| employee

In [10]:
attendance_tasks_df = attendance_tasks_df.withColumn(
    "productivity_score",
    col("tasks_completed") /
    col("effective_work_hours")
)
attendance_tasks_df.show()

+-----------+-------------+---------------+-------------------+-------------------+-------+--------------------+---------------+-----------+-----------------+--------------------+-------------------+
|employee_id|attendance_id|attendance_date|           clock_in|          clock_out|task_id|           task_name|tasks_completed|task_status|       work_hours|effective_work_hours| productivity_score|
+-----------+-------------+---------------+-------------------+-------------------+-------+--------------------+---------------+-----------+-----------------+--------------------+-------------------+
|        101|            1|     2026-06-01|2026-06-01 09:00:00|2026-06-01 18:00:00|     11|   attendance report|              7|  completed|              9.0|                 8.0|              0.875|
|        101|            1|     2026-06-01|2026-06-01 09:00:00|2026-06-01 18:00:00|      1|dashboard develop...|              8|  completed|              9.0|                 8.0|                1.0|


In [11]:
frequent_absentees = attendance_tasks_df.filter(
    col("effective_work_hours") < 7
)

frequent_absentees.show()

+-----------+-------------+---------------+-------------------+-------------------+-------+--------------------+---------------+-----------+-----------------+--------------------+-------------------+
|employee_id|attendance_id|attendance_date|           clock_in|          clock_out|task_id|           task_name|tasks_completed|task_status|       work_hours|effective_work_hours| productivity_score|
+-----------+-------------+---------------+-------------------+-------------------+-------+--------------------+---------------+-----------+-----------------+--------------------+-------------------+
|        104|            4|     2026-06-01|2026-06-01 10:15:00|2026-06-01 17:00:00|     14|    content creation|              2|  completed|             6.75|                5.75|0.34782608695652173|
|        104|            4|     2026-06-01|2026-06-01 10:15:00|2026-06-01 17:00:00|      4|social media camp...|              3|    pending|             6.75|                5.75| 0.5217391304347826|


In [12]:
underperformers = attendance_tasks_df.filter(
    col("tasks_completed") < 5
)

underperformers.show()

+-----------+-------------+---------------+-------------------+-------------------+-------+--------------------+---------------+-----------+-----------------+--------------------+-------------------+
|employee_id|attendance_id|attendance_date|           clock_in|          clock_out|task_id|           task_name|tasks_completed|task_status|       work_hours|effective_work_hours| productivity_score|
+-----------+-------------+---------------+-------------------+-------------------+-------+--------------------+---------------+-----------+-----------------+--------------------+-------------------+
|        102|            2|     2026-06-01|2026-06-01 09:20:00|2026-06-01 17:45:00|     12| employee onboarding|              4|  completed|8.416666666666666|   7.416666666666666| 0.5393258426966293|
|        104|            4|     2026-06-01|2026-06-01 10:15:00|2026-06-01 17:00:00|     14|    content creation|              2|  completed|             6.75|                5.75|0.34782608695652173|


In [13]:
from pyspark.sql.functions import avg

employee_report = attendance_tasks_df.groupBy(
    "employee_id"
).agg(
    avg("effective_work_hours").alias("average_work_hours"),
    avg("productivity_score").alias("average_productivity")
)

employee_report.show()

+-----------+------------------+--------------------+
|employee_id|average_work_hours|average_productivity|
+-----------+------------------+--------------------+
|        108| 6.916666666666667|  0.7238372093023255|
|        101| 7.958333333333332|  0.9424342105263158|
|        103| 8.166666666666666|  1.1632653061224492|
|        107|              8.25|  1.0304081632653062|
|        102|              7.25|  0.6210178453403834|
|        109| 8.333333333333332|     1.2001200120012|
|        105| 7.708333333333333|  0.9730014025245441|
|        110| 6.458333333333334|  0.6193806193806194|
|        106| 7.833333333333333|  1.0852292020373515|
|        104|               5.5| 0.45548654244306414|
+-----------+------------------+--------------------+



In [14]:
from pyspark.sql.functions import sum

task_summary = attendance_tasks_df.groupBy(
    "employee_id"
).agg(
    sum("tasks_completed").alias("total_tasks")
)

task_summary.show()

+-----------+-----------+
|employee_id|total_tasks|
+-----------+-----------+
|        108|         20|
|        101|         30|
|        103|         38|
|        107|         34|
|        102|         18|
|        109|         40|
|        105|         30|
|        110|          8|
|        106|         34|
|        104|         10|
+-----------+-----------+



In [15]:
employee_report.orderBy(
    col("average_productivity").desc()
).show()

+-----------+------------------+--------------------+
|employee_id|average_work_hours|average_productivity|
+-----------+------------------+--------------------+
|        109| 8.333333333333332|     1.2001200120012|
|        103| 8.166666666666666|  1.1632653061224492|
|        106| 7.833333333333333|  1.0852292020373515|
|        107|              8.25|  1.0304081632653062|
|        105| 7.708333333333333|  0.9730014025245441|
|        101| 7.958333333333332|  0.9424342105263158|
|        108| 6.916666666666667|  0.7238372093023255|
|        102|              7.25|  0.6210178453403834|
|        110| 6.458333333333334|  0.6193806193806194|
|        104|               5.5| 0.45548654244306414|
+-----------+------------------+--------------------+



In [16]:
employee_report.write.mode("overwrite").csv(
    "employee_productivity_report"
)

task_summary.write.mode("overwrite").csv(
    "employee_task_summary"
)